In [14]:
import math
class Value:
    def __init__(self,data,_children=(),_op='',label=''):
        self.data=data
        self._prev=set(_children)
        self._op=_op
        self.label=label
        self.grad=0.0
        self._backward=lambda:None

    def __repr__(self):
        return f"Value(data={self.data})"

    def __neg__(self):
        out = Value(-self.data, (self,), 'neg')
        def _backward():
            self.grad += -1.0 * out.grad
        out._backward = _backward
        return out

    def __add__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        out=Value(self.data+other.data,(self,other),'+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self,other):
        return self + other

    def __mul__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        out=Value(self.data*other.data,(self,other),'*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self,other):
        assert isinstance(other,(int,float))
        out=Value(self.data**other,(self,),f'**{other}')
        def _backward():
            self.grad += (other * (self.data ** (other - 1))) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        out=Value(self.data/other.data,(self,other),'/')
        def _backward():
            self.grad += out.grad * (1 / other.data)
            other.grad += out.grad * (-self.data / other.data**2)
        out._backward = _backward
        return out

    def exp(self):
        out=Value(math.exp(self.data),(self,), 'exp')
        def _backward():
            self.grad += out.grad * out.data
        out._backward = _backward
        return out

    def backward(self):
        topo=[]
        visited=set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        self.grad = 1.0
        build_topo(self)
        for v in reversed(topo):
            v._backward()


In [15]:
X = Value(3.0, label='x')
f = Value(1.0) / (Value(1.0) + (-X).exp())
f.label = 'sigmoid(x)'
f.backward()

analytical_grad = (-X).exp() / (Value(1.0) + (-X).exp())**2
print(f"f(x) = {f.data:.6f}")
print(f"df/dx = {X.grad:.6f}")
print(f"Analytical df/dx = {analytical_grad.data:.6f}")

f(x) = 0.952574
df/dx = 0.045177
Analytical df/dx = 0.045177
